# Tutorial 04: HLLSetStore — Persistent Storage with Derivation Tracking

This tutorial introduces the `HLLSetStore` module, which provides persistent storage for HLLSets with a derivation tracking system.

## What You'll Learn

1. **Architecture** — Base HLLSets vs. compound derivations
2. **Registering Base HLLSets** — Persistent storage
3. **Compound Operations** — Union, intersection, XOR with tracking
4. **On-the-fly Reconstruction** — Computing compounds from LUT
5. **Derivation Graph** — Exploring the dependency structure
6. **Storage Backends** — Custom persistence implementations

## Key Concepts

The store follows the principle:
> **"Store only base HLLSets; all compounds are generated on-the-fly"**

Architecture:
- **Base HLLSets**: Stored as serialized data (SHA1 → bytes)
- **Compound HLLSets**: NOT stored; reconstructed from LUT
- **HLLSet LUT**: Derivation graph mapping SHA1 → (operation, [operands])

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.hllset_store import (
    HLLSetStore, 
    HLLSetLUT, 
    Derivation, 
    Operation,
    InMemoryBackend
)
from core.hllset import HLLSet

## 1. Architecture Overview

The ring structure enables reconstruction:

```
H₄ = H₁ ∪ H₂  →  LUT["H₄"] = ("union", ["H₁", "H₂"])
H₅ = H₁ ⊕ H₃  →  LUT["H₅"] = ("xor", ["H₁", "H₃"])
```

This means:
- Base HLLSets consume storage
- Compound HLLSets only record their derivation
- Any HLLSet can be reconstructed from its ID

In [2]:
# Create a store
store = HLLSetStore(p_bits=10)

print(f"Store created with p_bits={store._p_bits}")
print(f"Using in-memory backend")

Store created with p_bits=10
Using in-memory backend


## 2. Registering Base HLLSets

Base HLLSets are the "source" data that gets persisted to storage.

In [3]:
# Create some HLLSets
hll_doc1 = HLLSet.from_batch(["hello", "world", "python"])
hll_doc2 = HLLSet.from_batch(["machine", "learning", "ai"])
hll_doc3 = HLLSet.from_batch(["python", "data", "science"])  # Overlaps with doc1

# Register as base HLLSets
id1 = store.register_base(hll_doc1, source="document_1")
id2 = store.register_base(hll_doc2, source="document_2")
id3 = store.register_base(hll_doc3, source="document_3")

print(f"Registered 3 base HLLSets:")
print(f"  doc1: {id1[:16]}...")
print(f"  doc2: {id2[:16]}...")
print(f"  doc3: {id3[:16]}...")

Registered 3 base HLLSets:
  doc1: 30b00c6e18959c06...
  doc2: c6c74026cbf51b2b...
  doc3: f30278736de7cd20...


In [4]:
# Check what's stored
print(f"\nStore statistics:")
print(f"  Base HLLSets: {len(list(store._lut.base_ids()))}")
print(f"  Total entries in LUT: {len(list(store._lut.all_ids()))}")


Store statistics:
  Base HLLSets: 3
  Total entries in LUT: 3


In [5]:
# Retrieve a base HLLSet
retrieved = store.get(id1)
print(f"\nRetrieved HLLSet:")
print(f"  Cardinality: {retrieved.cardinality():.0f}")
print(f"  Same as original: {retrieved.name == hll_doc1.name}")


Retrieved HLLSet:
  Cardinality: 4
  Same as original: True


## 3. Compound Operations

Compound operations create new HLLSets that are tracked but not stored.

| Operation | Method | Description |
|-----------|--------|-------------|
| Union | `union()` | A ∪ B |
| Intersect | `intersect()` | A ∩ B |
| Difference | `diff()` | A \ B |
| XOR | `xor()` | A ⊕ B |

In [6]:
# Union: doc1 ∪ doc2
id_union = store.union(id1, id2)
print(f"Union (doc1 ∪ doc2): {id_union[:16]}...")

# Check LUT entry
deriv = store._lut.lookup(id_union)
print(f"  Operation: {deriv.operation}")
print(f"  Operands: {[op[:12] for op in deriv.operands]}")

Union (doc1 ∪ doc2): 8c023865b4ea4c86...
  Operation: Operation.UNION
  Operands: ['30b00c6e1895', 'c6c74026cbf5']


In [7]:
# Intersection: doc1 ∩ doc3 (should capture "python")
id_intersect = store.intersect(id1, id3)
print(f"Intersection (doc1 ∩ doc3): {id_intersect[:16]}...")

# Retrieve and check
hll_intersect = store.get(id_intersect)
print(f"  Cardinality: {hll_intersect.cardinality():.0f} (should be ~1 for 'python')")

Intersection (doc1 ∩ doc3): 90159cca5d16c5dd...
  Cardinality: 2 (should be ~1 for 'python')


In [8]:
# XOR: doc1 ⊕ doc3 (symmetric difference)
id_xor = store.xor(id1, id3)
print(f"XOR (doc1 ⊕ doc3): {id_xor[:16]}...")

hll_xor = store.get(id_xor)
print(f"  Cardinality: {hll_xor.cardinality():.0f}")

XOR (doc1 ⊕ doc3): ac3f1cec6d19f645...
  Cardinality: 5


In [9]:
# Difference: doc1 \ doc3
id_diff = store.diff(id1, id3)
print(f"Difference (doc1 \\ doc3): {id_diff[:16]}...")

hll_diff = store.get(id_diff)
print(f"  Cardinality: {hll_diff.cardinality():.0f}")

Difference (doc1 \ doc3): a51c9340bf518ed3...
  Cardinality: 3


## 4. On-the-fly Reconstruction

Compound HLLSets are not stored — they're reconstructed from the LUT when requested.

In [10]:
# Create a chain of operations
# Step 1: doc1 ∪ doc2
id_step1 = store.union(id1, id2)

# Step 2: (doc1 ∪ doc2) ∪ doc3
id_step2 = store.union(id_step1, id3)

print(f"Compound chain:")
print(f"  Step 1 (doc1 ∪ doc2): {id_step1[:12]}...")
print(f"  Step 2 ((doc1 ∪ doc2) ∪ doc3): {id_step2[:12]}...")

Compound chain:
  Step 1 (doc1 ∪ doc2): 8c023865b4ea...
  Step 2 ((doc1 ∪ doc2) ∪ doc3): 44524fadd6a3...


In [11]:
# Retrieve compound - reconstruction happens automatically
hll_compound = store.get(id_step2)

print(f"\nReconstructed compound:")
print(f"  Cardinality: {hll_compound.cardinality():.0f}")
print(f"  (All three documents combined)")


Reconstructed compound:
  Cardinality: 9
  (All three documents combined)


In [12]:
# Verify reconstruction matches direct computation
direct = HLLSet.merge([hll_doc1, hll_doc2, hll_doc3])
print(f"Direct merge cardinality: {direct.cardinality():.0f}")
print(f"Reconstructed cardinality: {hll_compound.cardinality():.0f}")

Direct merge cardinality: 9
Reconstructed cardinality: 9


## 5. Derivation Graph

The LUT maintains a derivation graph that tracks dependencies.

In [13]:
# View all IDs in the store
all_ids = list(store._lut.all_ids())
print(f"Total IDs in LUT: {len(all_ids)}")

# Separate base and compound
base_ids = list(store._lut.base_ids())
compound_ids = [i for i in all_ids if i not in base_ids]

print(f"  Base HLLSets: {len(base_ids)}")
print(f"  Compound HLLSets: {len(compound_ids)}")

Total IDs in LUT: 8
  Base HLLSets: 3
  Compound HLLSets: 5


In [14]:
# Explore derivation depth
print("\nDerivation depths:")
for hid in all_ids[:8]:  # First 8
    depth = store._lut.derivation_depth(hid)
    deriv = store._lut.lookup(hid)
    op_name = deriv.operation.value if deriv else "?"
    print(f"  {hid[:12]}...: depth={depth}, op={op_name}")


Derivation depths:
  30b00c6e1895...: depth=0, op=base
  c6c74026cbf5...: depth=0, op=base
  f30278736de7...: depth=0, op=base
  8c023865b4ea...: depth=1, op=union
  90159cca5d16...: depth=1, op=intersect
  ac3f1cec6d19...: depth=1, op=xor
  a51c9340bf51...: depth=1, op=diff
  44524fadd6a3...: depth=2, op=union


In [15]:
# Find base HLLSets contributing to a compound
bases = store._lut.get_bases(id_step2)
print(f"\nBase HLLSets for compound {id_step2[:12]}...:")
for base_id in bases:
    deriv = store._lut.lookup(base_id)
    source = deriv.metadata.get("source", "unknown") if deriv else "?"
    print(f"  {base_id[:12]}... (source: {source})")


Base HLLSets for compound 44524fadd6a3...:
  30b00c6e1895... (source: document_1)
  c6c74026cbf5... (source: document_2)
  f30278736de7... (source: document_3)


In [16]:
# Find compounds that depend on a base
dependents = store._lut.get_dependents(id1)
print(f"\nCompounds depending on doc1 ({id1[:12]}...):")
for dep_id in list(dependents)[:5]:  # First 5
    deriv = store._lut.lookup(dep_id)
    print(f"  {dep_id[:12]}... ({deriv.operation.value})")


Compounds depending on doc1 (30b00c6e1895...):
  90159cca5d16... (intersect)
  a51c9340bf51... (diff)
  ac3f1cec6d19... (xor)
  8c023865b4ea... (union)


## 6. Existence Checking and Iteration

The store supports checking existence and iterating over stored HLLSets.

In [17]:
# Check existence
print(f"ID {id1[:12]}... exists: {store.exists(id1)}")
print(f"Random ID exists: {store.exists('0' * 40)}")

ID 30b00c6e1895... exists: True
Random ID exists: False


In [18]:
# Check if base or compound
print(f"\nID classification:")
print(f"  doc1 is base: {store._lut.is_base(id1)}")
print(f"  union is base: {store._lut.is_base(id_union)}")


ID classification:
  doc1 is base: True
  union is base: False


## 7. Storage Backends

The store uses a pluggable storage backend. You can implement custom backends for different persistence needs.

In [19]:
# View InMemoryBackend interface
backend = InMemoryBackend()

# Basic operations
backend.put("key1", b"value1")
backend.put("key2", b"value2")

print(f"Get 'key1': {backend.get('key1')}")
print(f"Exists 'key1': {backend.exists('key1')}")
print(f"Exists 'key3': {backend.exists('key3')}")
print(f"Keys: {list(backend.keys())}")

Get 'key1': b'value1'
Exists 'key1': True
Exists 'key3': False
Keys: ['key1', 'key2']


In [20]:
# Create store with custom backend
custom_backend = InMemoryBackend()
custom_store = HLLSetStore(backend=custom_backend, p_bits=10)

# Use the store
hll = HLLSet.from_batch(["custom", "backend", "test"])
cid = custom_store.register_base(hll, source="test")

print(f"Registered in custom store: {cid[:16]}...")
print(f"Backend size: {len(custom_backend)} entries")

Registered in custom store: a32277a56a9098c8...
Backend size: 0 entries


## 8. Practical Example: Document Corpus Analysis

Let's simulate analyzing a document corpus with the store.

In [21]:
# Simulate a document corpus
corpus = {
    "intro_ml": ["machine", "learning", "data", "model", "training"],
    "deep_learning": ["neural", "network", "deep", "learning", "layer"],
    "nlp_basics": ["natural", "language", "processing", "text", "nlp"],
    "python_data": ["python", "pandas", "data", "analysis", "dataframe"],
    "ml_python": ["machine", "learning", "python", "scikit", "model"],
}

# Create store and register documents
corpus_store = HLLSetStore(p_bits=10)
doc_ids = {}

for doc_name, tokens in corpus.items():
    hll = HLLSet.from_batch(tokens)
    doc_id = corpus_store.register_base(hll, source=doc_name)
    doc_ids[doc_name] = doc_id
    print(f"Registered '{doc_name}': {doc_id[:12]}...")

Registered 'intro_ml': d1e63f02c568...
Registered 'deep_learning': e2700347cd33...
Registered 'nlp_basics': fe65e55480b5...
Registered 'python_data': d23ee17499b8...
Registered 'ml_python': a5b2393ae995...


In [22]:
# Find documents with overlapping topics
print("\nTopic overlaps:")

pairs = [
    ("intro_ml", "deep_learning"),
    ("intro_ml", "ml_python"),
    ("nlp_basics", "python_data"),
]

for doc_a, doc_b in pairs:
    id_intersect = corpus_store.intersect(doc_ids[doc_a], doc_ids[doc_b])
    hll_common = corpus_store.get(id_intersect)
    print(f"  {doc_a} ∩ {doc_b}: ~{hll_common.cardinality():.0f} common terms")


Topic overlaps:
  intro_ml ∩ deep_learning: ~2 common terms
  intro_ml ∩ ml_python: ~5 common terms
  nlp_basics ∩ python_data: ~0 common terms


In [23]:
# Create topic clusters
# ML cluster: intro_ml ∪ deep_learning ∪ ml_python
ml_union_1 = corpus_store.union(doc_ids["intro_ml"], doc_ids["deep_learning"])
ml_cluster = corpus_store.union(ml_union_1, doc_ids["ml_python"])

# Data cluster: python_data
data_cluster = doc_ids["python_data"]

print(f"\nCluster sizes:")
print(f"  ML cluster: ~{corpus_store.get(ml_cluster).cardinality():.0f} unique terms")
print(f"  Data cluster: ~{corpus_store.get(data_cluster).cardinality():.0f} unique terms")


Cluster sizes:
  ML cluster: ~12 unique terms
  Data cluster: ~6 unique terms


In [24]:
# Find ML-specific terms (not in data cluster)
ml_specific = corpus_store.diff(ml_cluster, data_cluster)
hll_specific = corpus_store.get(ml_specific)

print(f"\nML-specific terms: ~{hll_specific.cardinality():.0f}")


ML-specific terms: ~10


In [ ]:
# Explore the derivation graph
print(f"\nDerivation graph for ML cluster:")
print(f"  Depth: {corpus_store._lut.derivation_depth(ml_cluster)}")
bases = corpus_store._lut.get_bases(ml_cluster)
print(f"  Contributing documents:")
for bid in bases:
    deriv = corpus_store._lut.lookup(bid)
    source = deriv.metadata.get("source", "?") if deriv else "?"
    print(f"    - {source}")

## 9. Derivation Object Details

Let's explore the `Derivation` object that tracks how HLLSets are constructed.

In [25]:
# View a derivation
deriv = corpus_store._lut.lookup(ml_cluster)

print(f"Derivation for ML cluster:")
print(f"  Operation: {deriv.operation}")
print(f"  Operands: {len(deriv.operands)}")
print(f"  Timestamp: {deriv.timestamp:.2f}")
print(f"  Metadata: {deriv.metadata}")
print(f"  Is base: {deriv.is_base()}")

Derivation for ML cluster:
  Operation: Operation.UNION
  Operands: 2
  Timestamp: 1775929343.49
  Metadata: {}
  Is base: False


In [26]:
# Serialize derivation
deriv_dict = deriv.to_dict()
print(f"\nSerialized derivation:")
import json
print(json.dumps(deriv_dict, indent=2))


Serialized derivation:
{
  "operation": "union",
  "operands": [
    "477beb9447bcceb7c2de277403c1efc6e6cf37b1",
    "a5b2393ae9951d8ed6918ae2f75fb39e3e3a1a5a"
  ],
  "timestamp": 1775929343.4893553,
  "metadata": {}
}


In [27]:
# Deserialize
deriv_restored = Derivation.from_dict(deriv_dict)
print(f"\nRestored derivation:")
print(f"  Operation: {deriv_restored.operation}")
print(f"  Same as original: {deriv_restored.operation == deriv.operation}")


Restored derivation:
  Operation: Operation.UNION
  Same as original: True


## Summary

In this tutorial, you learned:

1. **Architecture** — Base HLLSets stored, compounds tracked in LUT
2. **Registration** — `register_base()` for persistent storage
3. **Operations** — `union()`, `intersect()`, `diff()`, `xor()` with tracking
4. **Reconstruction** — `get()` computes compounds on-the-fly
5. **Derivation Graph** — Dependencies, depth, base discovery
6. **Storage Backends** — Pluggable persistence

### Key Benefits

- **Storage Efficiency**: Only base HLLSets consume storage
- **Reproducibility**: Complete derivation history tracked
- **Flexibility**: Any compound can be reconstructed
- **Extensibility**: Custom backends for different persistence needs

### Completed Tutorials

✅ Tutorial 01: HLLSet — Core operations  
✅ Tutorial 02: HLLTensor — 2D tensor view  
✅ Tutorial 03: HLLLattice — Temporal lattice  
✅ Tutorial 04: HLLSetStore — Persistent storage